# IHC-Global analysis: data collection and aggregation

First step in IHC analysis. The workflow is as follows:

1 - Run this script, which combines all recordingsd in three files, master , alltraces, allCorrTraces, which contains all cells and all traces/correlationTraces (correlationTraces are timeseries representing the local correlation of the pixels of one ROI over time. Note if you rerun this code, by default it copy over the cells already analysed from the old file (produced by traceExplorer, see below) and just adds the new ones.

2 - Run the IHC-Deconvolution-Cascade notebook to predict the spike probability using cascade for all the traces. 

3 - Use the traceExplorer GUI (in src) to open, the master , alltraces, allCorrTraces and label the peaks semiautomatically.

4 - Run the IHC-GlobalAnalysis-part2 - dataProcessing for further analysis (e.g., selection of high quality peaks etc..)

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%pylab
%load_ext autoreload
%autoreload 2
%matplotlib inline
import pandas as pd
import os
from scipy.signal import savgol_filter,find_peaks
from scipy.signal import argrelextrema,argrelmin,argrelmax
import sys
sys.path.append('../../src')
import seaborn as sns

import ast 
from movieTools import thorlabsFile
from tifffile import imread

parameterFolder = '../../parameters/'

fileHeader = [
'alpha9'
]


#Load and concatenate all the files with the IHC experiments. Create unique mouse IDs by appending the strain to the mouse number
alldata = []
for h in fileHeader:
    inputFilename = os.path.join('../../',h)+'.xlsx'

    alldata1 = pd.read_excel(inputFilename)
    alldata.append(alldata1[alldata1['discard']!=1])


alldata = pd.concat(alldata, ignore_index=True)
alldata = alldata[~alldata['Folder'].isna()]
alldata = alldata.reset_index()

alldata = alldata.sort_values(['Age','Mouse ID','Folder']).reset_index()


#Folders to load and save the data from
resultsFOlder = '../../analysis_results/'
fileHeader = 'alpha9'

inputFilename = os.path.join(resultsFOlder,fileHeader)+'.xlsx'

tempFilename = '../../analysis_results/master_temp.csv'

# Some functions first

In [ ]:
from skimage import measure
from movieTools import determineCellTypes, determineCentroids
from scipy.interpolate import interp1d
from traceUtilities import fillMissingValues,rollingMedianCorrection, rollingPercentileCorrection, detrend_z_score

def extractTraces(folder,fps = 1,returndFF0 = True,returnCorrelations = True, fillMissingValue = False):
    filename = os.path.join(folder,'processedMovies','traces.csv')
    traces = pd.read_csv(filename,index_col=0)
    x = traces['Frame']/fps
    traces = traces.drop('Frame',axis=1)
    if fillMissingValue:
        traces = fillMissingValues(traces)
    stop = int(traces.shape[0]/2)
    f0 = traces.iloc[:stop,:].quantile(0.3)
    
    dff0 = (traces - f0)/f0



    if returnCorrelations:
        filename = os.path.join(folder,'processedMovies','traces_localCorr.csv')
        correlations = pd.read_csv(filename,index_col=0)
        if returndFF0:
            return x,dff0,correlations
        else:
            return x,traces,correlations

    else:
        if returndFF0:
            return x,dff0
        else:
            return x,traces

def extractImages(folder):
    filename = os.path.join(folder,'processedMovies','Masks.tif')
    filename2 =  os.path.join(folder,'processedMovies','Annotations.tif')
    filename3 =  os.path.join(folder,'processedMovies','Avg.tif')
    masks = imread(filename)
    annotations = imread(filename2)
    avg = imread(filename3)

    return (avg,masks,annotations)

def imshowImage(avg,masks,annotations, highlighCell = None):
    if highlighCell is None:
        masks2 = masks.copy().astype(float)
        masks2[masks==0] = np.nan
        figure()
        imshow(avg,cmap=cm.gray)
        masks2 = annotations.copy().astype(float)
        masks2[masks==0] = np.nan
        imshow(masks2)
    else:
        masks2 = masks.copy().astype(float)
        masks2[masks==0] = np.nan
        figure()
        imshow(avg,cmap=cm.gray)
        masks2 = annotations.copy().astype(float)
        masks2[masks!=highlighCell] = np.nan
        imshow(masks2)

def renderAnnotations(tb,avg,masks,annotations):
    try:
        l = tb.app.layers['Avg']
        l.data = avg
    except:
        tb.app.add_image(avg,name='Avg',visible=False)
    try:
        l = tb.app.layers['Masks']
        l.data = masks
    except:
        tb.app.add_labels(masks,name='Masks',visible=False)
    
    try:
        l = tb.app.layers['Annotations']
        l.data = annotations
    except:
        tb.app.add_labels(annotations,name='Annotations',visible=False)

from scipy.signal import periodogram
def noise_amplitude(trace):
    y = periodogram(trace,window='flattop',scaling='spectrum')[1]
    halfL = int(len(y)/2)
    return np.sqrt(y[halfL:].mean())*np.sqrt(2)


# Create a database of all the traces and of all the identified events, or load from a file

In [ ]:
# Change drive automatically by probing candidate locations
from traceUtilities import determineLocalDrive

localDrive = determineLocalDrive(
    alldata,
    folderColumn='Folder',
    candidates=['/media/marcotti-lab', 'D:', 'E:', 'F:', 'Z:'],
    nRandomChecks=3,
    randomState=0,
 )
print(f'Using localDrive: {localDrive}')
alldata['Folder'] = localDrive + alldata['Folder'].str[2:].str.replace('\\','/')

In [ ]:
# Ensure Number in sequence is contiguous within each Independent recordings number
original_sequence = alldata['Number in sequence'].copy()
new_sequence = pd.Series(index=alldata.index, dtype=float)

for rec_id, idx in alldata.groupby('Independent recordings number').groups.items():
    ordered_idx = alldata.loc[idx, 'Number in sequence'].sort_values().index
    new_sequence.loc[ordered_idx] = np.arange(1, len(ordered_idx) + 1)

alldata['Number in sequence'] = new_sequence.astype(int)

# Report which groups were changed
changed_mask = original_sequence != alldata['Number in sequence']
changed_groups = alldata.loc[changed_mask, 'Independent recordings number'].dropna().unique()

if len(changed_groups) == 0:
    print('No sequence gaps found.')
else:
    print(f'Renumbered groups to contiguous sequence: {changed_groups.tolist()}')

In [ ]:
from tqdm.auto import tqdm
import tifffile
def createMaster(alldata, inputFilename=inputFilename, overrideSave=False):
    '''
    This function will go through the recordings and create a master where every row is a ROI. 
    Initially it will load a previous file (inputfilename) if it exist, and add ROIs to it only if they don't exist already. 
    If overrideSave is true it will save the files even if no new cells have been added, otherwise it will save the file only
    if new cells are present
    '''
    try:
        oldmaster = pd.read_excel(inputFilename)
        oldmaster['Peak positions'] = oldmaster['Peak positions'].apply(ast.literal_eval)
    except FileNotFoundError:
        print('No result file found')
        oldmaster = pd.DataFrame(columns=['Cell ID'])
    
    new_cells = 0
    master_rows = []
    trace_columns = []
    corr_columns = []

    pbar = tqdm(
        alldata.iterrows(),
        total=len(alldata),
        desc='Processing recordings',
        unit='rec',
        dynamic_ncols=True
    )
    for i,el in pbar:
        current_folder = str(el['Folder']).split('/')[-1]
        pbar.set_postfix_str(current_folder)

        try: 
            x,t = extractTraces(el['Folder'],el['fps'], fillMissingValue= True, returnCorrelations=False)
            correlations = None
            avg,masks,annotations = extractImages(el['Folder'])
            celltypes = determineCellTypes(masks,annotations)
            centroids = determineCentroids(masks) 
            if t.shape[1]!= celltypes.size:
                print(t.shape)
                print(celltypes.size)
                print(el['Folder']+' Error: mismatch number of cells')
                break
                
            for j in range(t.shape[1]):
                cellID = el['Date'] + '_' + el['Strain'] + '_' + str(el['Mouse ID'])+'_'+el['Folder'].split('/')[-1].split('\\')[-1] + '_' + t.columns[j]

                trace_columns.append(t.iloc[:, j].rename(cellID))
                if correlations is not None:
                    corr_columns.append(correlations.iloc[:, j].rename(cellID))

                this_row = {}
                this_row['Date'] = el['Date']
                this_row['Folder'] = el['Folder']
                this_row['RoiN'] = t.columns[j]
                this_row['Cell type'] = celltypes[t.columns[j]]
                this_row['Mouse ID'] = el['Mouse ID']
                this_row['Strain'] = el['Strain']
                this_row['Age'] = el['Age']
                this_row['fps'] = el['fps']

                if cellID not in oldmaster['Cell ID'].values: # If the cell already exists in the db, don't load it.
                    this_row['Peak prominence'] = 4.01
                    this_row['Peak min distance'] = 150 #Empirical observation.
                    this_row['Peak min height'] = 0
                    this_row['Peak correlation'] = 0
                    this_row['Use MAD z-score'] = True
                    this_row['Min duration (s)'] = 0.5

                    # Run a first pass of peak finding
                    fps = el['fps']
                    minDuration = 0.5*fps
                    distance = this_row['Peak min distance'] 
                    #detrent traces
                    #corrected = rollingMedianCorrection(t.iloc[:,j], rollingN=2000)
                    # corrected = rollingPercentileCorrection(t.iloc[:,j],windowFrames = 2000)
                    # detect_trace = mad_zscore(corrected)
                    # detect_trace = savgol_filter(detect_trace,21, 1)
                    detect_trace = detrend_z_score(t.iloc[:,j], rollingN=2000, savgol_order=21)
                    peaks, _ = find_peaks(detect_trace, height=0, prominence=4.01, distance=distance)
                    # # Make sure the peaks are aligned with the original trace by hooking it to a maximum in the original trace within a windows of "distance" from where it was found
                    # aligned_peaks = []
                    # for peak in peaks:
                    #     window_start = max(0, peak - distance)
                    #     window_end = min(len(t.iloc[:, j]), peak + distance + 1)
                    #     max_in_window = np.argmax(t.iloc[window_start:window_end, j]) + window_start
                    #     aligned_peaks.append(max_in_window)
                    # peaks = aligned_peaks

                    this_row['Peak positions'] = list(set(peaks))

                    new_cells = new_cells + 1

                else:
                    previous_row = oldmaster.loc[oldmaster['Cell ID']==cellID]
                    if previous_row.shape[0]!=1:
                        print('Problem')

                    this_row['Peak prominence'] = previous_row['Peak prominence'].values[0]
                    this_row['Peak min distance'] = previous_row['Peak min distance'].values[0]
                    this_row['Peak min height'] = previous_row['Peak min height'].values[0]
                    this_row['Peak correlation'] = previous_row['Peak correlation'].values[0]
                    this_row['Use MAD z-score'] = previous_row['Use MAD z-score'].values[0]
                    this_row['Min duration (s)'] = previous_row['Min duration (s)'].values[0]
                    this_row['Peak positions'] = previous_row['Peak positions'].values[0] #list(find_peaks(t.iloc[:,j],prominence=0.2)[0])

                this_row['Centroid x'] = centroids[t.columns[j]][0]
                this_row['Centroid y'] = centroids[t.columns[j]][1]
                this_row['Centroid x um'] = centroids[t.columns[j]][0] * el['um/pixel']
                this_row['Centroid y um'] = centroids[t.columns[j]][1] * el['um/pixel']

                this_row['Independent recordings number'] = el['Independent recordings number']
                this_row['Number in sequence'] = el['Number in sequence']

                this_row['Cell ID'] = cellID

                master_rows.append(this_row)

        except FileNotFoundError:
            print('File not found: '+el['Folder'])
            
    print('Added '+str(new_cells)+' cells.')

    master = pd.DataFrame(master_rows)
    alltraces = pd.concat(trace_columns, axis=1) if len(trace_columns) > 0 else pd.DataFrame()
    allCorrTraces = pd.concat(corr_columns, axis=1) if len(corr_columns) > 0 else pd.DataFrame()

    master['Discard'] = np.nan

    #  Add the information regarding the independent recordings number and the unique cell id across recordings
    #Add the unique indentifier for matched experiments
    for folder in master['Folder'].unique():
        #print(folder)
        master.loc[master['Folder']==folder,'Independent recordings ID'] = master.loc[master['Folder']==folder,'Strain'].values[0] + '_' + str(master.loc[master['Folder']==folder,'Independent recordings number'].values[0])   
      

    # Assign to each roi the same "Matched number" so they can be traced in different recordings. This is based on the MatchingMasks.tif file 

    print('Determining matched ROIs in different recordings...')
    for folder in tqdm(master['Folder'].unique()):
        el = master[master['Folder']==folder]
        filename = os.path.join(folder,'processedMovies','Masks.tif')
        filename2 =  os.path.join(folder,'processedMovies','MatchingMasks.tif')
        filename3 =  os.path.join(folder,'processedMovies','Avg.tif')
        filename4 =  os.path.join(folder,'processedMovies','Annotations.tif')
        try:
            matchinmasks = tifffile.imread(filename2)
            masks = tifffile.imread(filename)
            annotations = tifffile.imread(filename4)

            if matchinmasks.shape != masks.shape:
                print('The matching masks and the masks have different shapes for folder '+folder)
                continue
            else:
                celltypes = determineCellTypes(masks,annotations)
                for j, row in el.iterrows():
                    if celltypes[row['RoiN']]==1:
                        roin = int(row['RoiN'].split('_')[1])
                        master.loc[j,'Matched RoiN'] = np.median(matchinmasks[masks==roin])
                        if np.median(matchinmasks[masks==roin])%1 != 0 :
                            print('Some problems here!!!!')
                    else:
                        master.loc[j,'Matched RoiN'] = np.nan
        

        except FileNotFoundError:
            masks = tifffile.imread(filename)
            annotations = tifffile.imread(filename4)
            celltypes = determineCellTypes(masks,annotations)
            cumRoiNumber = 1
            for j, row in el.iterrows():

                if celltypes[row['RoiN']]==1:
                    roin = int(row['RoiN'].split('_')[1])
                    master.loc[j,'Matched RoiN'] = cumRoiNumber
                    cumRoiNumber = cumRoiNumber+1
                    
                else:
                    master.loc[j,'Matched RoiN'] = np.nan


    #Save the file again if new cells are found
    if new_cells!=0 or overrideSave:
        master.to_excel(inputFilename)
        alltraces.to_parquet(os.path.join(resultsFOlder,f'{fileHeader}_events_alltraces.parquet'))
        allCorrTraces.to_parquet(os.path.join(resultsFOlder,f'{fileHeader}_events_allCorrTraces.parquet'))
    return master, alltraces, allCorrTraces


# Load directly the results from a previously saved file or recompute the master,traces and corrTraces from scratch (uncomment as required)

In [ ]:
#Recreate from scratch. If a inputfileName is provided, previously identified calcium events will be carried over 
master , alltraces, allCorrTraces = createMaster(alldata,overrideSave=False)